In [1]:
import numpy as np
from PIL import Image
import torch
import torchvision

import matplotlib.pyplot as plt
from build_sam import sam_model_registry
import os



/Users/abdu/miniconda3/envs/sam_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# img resolution
img_resolution = 1024

# Select Proper SAM Size you want
sam = sam_model_registry['sam_encoder_b'](checkpoint='../segment-anything/checkpoints/sam_vit_b_image_encoder.pth', custom_img_size=img_resolution)

In [5]:
sam

ImageEncoderViT(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
  )
  (blocks): ModuleList(
    (0-11): 12 x Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (proj): Linear(in_features=768, out_features=768, bias=True)
      )
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (mlp): MLPBlock(
        (lin1): Linear(in_features=768, out_features=3072, bias=True)
        (lin2): Linear(in_features=3072, out_features=768, bias=True)
        (act): GELU(approximate='none')
      )
    )
  )
  (neck): Sequential(
    (0): Conv2d(768, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (1): LayerNorm2d()
    (2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (3): LayerNorm2d()
  )
)

In [5]:
rand_input = torch.rand(1, 3, 256, 256)
rand_output = sam(rand_input)
print(rand_output.shape)

torch.Size([1, 256, 16, 16])


In [3]:
student_model = sam_model_registry['student_encoder']()
student_model

TinyViT(
  (patch_embed): PatchEmbed(
    (seq): Sequential(
      (0): Conv2d_BN(
        (c): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): GELU(approximate='none')
      (2): Conv2d_BN(
        (c): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
  )
  (layers): ModuleList(
    (0): ConvLayer(
      (blocks): ModuleList(
        (0-1): 2 x MBConv(
          (conv1): Conv2d_BN(
            (c): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (bn): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          )
          (act1): GELU(approximate='none')
          (conv2): Conv2d_BN(
            (c): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), pa

In [7]:
student_output = student_model(rand_input)
print(student_output.shape)

torch.Size([1, 256, 16, 16])


In [8]:
# measure the MSE between the two outputs
mse = torch.nn.MSELoss()
loss = mse(rand_output, student_output)
print(loss.item())

1.02704918384552
